In [14]:
def append_jsonl(record, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def read_jsonl(path: Path):
    if not path.exists():
        return []
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                pass
    return rows


def normalize_id_series(series):
    return (
        series.astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )


def load_seen_ids():
    seen_ids = set()

    if BATCH_CSV.exists():
        try:
            old_df = pd.read_csv(BATCH_CSV, dtype={"매물ID": str})
            if "매물ID" in old_df.columns:
                ids = normalize_id_series(old_df["매물ID"].dropna())
                seen_ids.update(ids.tolist())
        except Exception as e:
            print(f"[WARN] 기존 BATCH_CSV 읽기 실패: {e}")

    if SUCCESS_LOG_JSONL.exists():
        try:
            success_rows = read_jsonl(SUCCESS_LOG_JSONL)
            for row in success_rows:
                car_id = row.get("매물ID")
                if car_id:
                    seen_ids.add(str(car_id).strip())
        except Exception as e:
            print(f"[WARN] SUCCESS_LOG_JSONL 읽기 실패: {e}")

    return seen_ids


def rebuild_batch_csv():
    frames = []

    if BATCH_CSV.exists():
        try:
            old_df = pd.read_csv(BATCH_CSV, dtype={"매물ID": str})
            frames.append(old_df)
        except Exception as e:
            print(f"[WARN] 기존 BATCH_CSV 재읽기 실패: {e}")

    success_rows = read_jsonl(SUCCESS_LOG_JSONL)
    if success_rows:
        frames.append(pd.DataFrame(success_rows))

    if not frames:
        return pd.DataFrame()

    df = pd.concat(frames, ignore_index=True, sort=False)

    if "매물ID" in df.columns:
        df["매물ID"] = normalize_id_series(df["매물ID"])
        df = df[df["매물ID"].notna() & (df["매물ID"] != "")]
        df = df.drop_duplicates(subset=["매물ID"], keep="first")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(BATCH_CSV, index=False, encoding="utf-8-sig")
    return df


def rebuild_error_csv():
    error_rows = read_jsonl(ERROR_LOG_JSONL)
    if not error_rows:
        return pd.DataFrame()

    df = pd.DataFrame(error_rows)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(ERROR_CSV, index=False, encoding="utf-8-sig")
    return df


def export_final_json():
    if not BATCH_CSV.exists():
        return

    df = pd.read_csv(BATCH_CSV, dtype={"매물ID": str})
    records = df.where(pd.notnull(df), None).to_dict(orient="records")

    with open(FINAL_JSON, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


def validate_row(row):
    key_fields = ["차량명", "현재가격_만원", "연식", "주행거리_km"]
    filled = 0
    for k in key_fields:
        v = row.get(k)
        if v is not None and str(v).strip() not in ["", "nan", "None"]:
            filled += 1

    if filled < 2:
        raise RuntimeError("핵심 필드 부족: 에러 페이지 또는 파싱 실패 가능성")

    return True

In [15]:
import re
import time
import json
import random
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# =========================
# 설정
# =========================
TARGET_TOTAL = 5000            # 최종 누적 목표 건수
SAVE_EVERY = 20                # 성공 20건마다 CSV 재생성
RESTART_EVERY = 50             # 상세페이지 50건 처리마다 드라이버 재시작
MAX_LISTING_PAGES = 30         # 목록 페이지 최대 탐색 수
MAX_SCROLLS_PER_PAGE = 15
HEADLESS = True                # 디버깅할 때만 False
NEW_LINK_BUFFER = 150          # 상세 실패 대비 링크 여유 확보

BASE_SAVE_DIR = Path("encar_kia_all")
HTML_DIR = BASE_SAVE_DIR / "html"
OUTPUT_DIR = BASE_SAVE_DIR / "output"

BATCH_CSV = OUTPUT_DIR / "kia_detail_batch.csv"
ERROR_CSV = OUTPUT_DIR / "kia_detail_errors.csv"
FINAL_JSON = OUTPUT_DIR / "kia_detail_final.json"

SUCCESS_LOG_JSONL = OUTPUT_DIR / "kia_detail_success_log.jsonl"
ERROR_LOG_JSONL = OUTPUT_DIR / "kia_detail_error_log.jsonl"

SEARCH_URL = (
    "https://car.encar.com/list/car?page=1&search=%7B%22type%22%3A%22car%22%2C"
    "%22action%22%3A%22(And.Hidden.N._.MultiView2Hidden.N._.(C.CarType.Y._.Manufacturer.%EA%B8%B0%EC%95%84.))%22%2C"
    "%22title%22%3A%22%EA%B8%B0%EC%95%84%22%2C%22toggle%22%3A%7B%7D%2C"
    "%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)

In [16]:
# =========================
# 공통 유틸
# =========================
def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip() if text else ""


def find_first(patterns, text, cast=None, flags=re.S):
    for pat in patterns:
        m = re.search(pat, text, flags)
        if m:
            val = m.group(1) if m.lastindex else m.group(0)
            if cast:
                try:
                    return cast(val)
                except Exception:
                    return val
            return val
    return None


def to_int(val):
    if val is None:
        return None
    return int(str(val).replace(",", "").strip())


def parse_int_from_text(text):
    if not text:
        return None
    m = re.search(r"([\d,]+)", text)
    return int(m.group(1).replace(",", "")) if m else None


def parse_json_array_str(text):
    if not text:
        return []
    try:
        return json.loads(text)
    except Exception:
        return []


def extract_car_id(url):
    m = re.search(r"/cars/detail/(\d+)", url)
    return m.group(1) if m else None


def build_full_trim(model, grade, detail):
    parts = []
    for p in [model, grade, detail]:
        p = normalize_text(p)
        if p and p not in parts:
            parts.append(p)
    return " ".join(parts) if parts else None


def convert_year_from_embedded(year_month, form_year, fallback_text):
    if form_year:
        try:
            return int(form_year)
        except Exception:
            pass
    if year_month and len(year_month) >= 4:
        try:
            return int(year_month[:4])
        except Exception:
            pass
    if fallback_text:
        m = re.search(r"(\d{2})/", fallback_text)
        if m:
            return 2000 + int(m.group(1))
    return None


def safe_sleep(a=1.8, b=3.2):
    time.sleep(random.uniform(a, b))

In [17]:
# =========================
# 파일 저장 / resume / 검증
# =========================
def append_jsonl(record, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def read_jsonl(path: Path):
    if not path.exists():
        return []
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                pass
    return rows


def normalize_id_series(series):
    return (
        series.astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )


def load_seen_ids():
    seen_ids = set()

    if BATCH_CSV.exists():
        try:
            old_df = pd.read_csv(BATCH_CSV, dtype={"매물ID": str})
            if "매물ID" in old_df.columns:
                ids = normalize_id_series(old_df["매물ID"].dropna())
                seen_ids.update(ids.tolist())
        except Exception as e:
            print(f"[WARN] 기존 BATCH_CSV 읽기 실패: {e}")

    if SUCCESS_LOG_JSONL.exists():
        try:
            success_rows = read_jsonl(SUCCESS_LOG_JSONL)
            for row in success_rows:
                car_id = row.get("매물ID")
                if car_id:
                    seen_ids.add(str(car_id).strip())
        except Exception as e:
            print(f"[WARN] SUCCESS_LOG_JSONL 읽기 실패: {e}")

    return seen_ids


def rebuild_batch_csv():
    frames = []

    if BATCH_CSV.exists():
        try:
            old_df = pd.read_csv(BATCH_CSV, dtype={"매물ID": str})
            frames.append(old_df)
        except Exception as e:
            print(f"[WARN] 기존 BATCH_CSV 재읽기 실패: {e}")

    success_rows = read_jsonl(SUCCESS_LOG_JSONL)
    if success_rows:
        frames.append(pd.DataFrame(success_rows))

    if not frames:
        return pd.DataFrame()

    df = pd.concat(frames, ignore_index=True, sort=False)

    if "매물ID" in df.columns:
        df["매물ID"] = normalize_id_series(df["매물ID"])
        df = df[df["매물ID"].notna() & (df["매물ID"] != "")]
        df = df.drop_duplicates(subset=["매물ID"], keep="first")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(BATCH_CSV, index=False, encoding="utf-8-sig")
    return df


def rebuild_error_csv():
    error_rows = read_jsonl(ERROR_LOG_JSONL)
    if not error_rows:
        return pd.DataFrame()

    df = pd.DataFrame(error_rows)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(ERROR_CSV, index=False, encoding="utf-8-sig")
    return df


def export_final_json():
    if not BATCH_CSV.exists():
        return

    df = pd.read_csv(BATCH_CSV, dtype={"매물ID": str})
    records = df.where(pd.notnull(df), None).to_dict(orient="records")

    with open(FINAL_JSON, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


def validate_row(row):
    key_fields = ["차량명", "현재가격_만원", "연식", "주행거리_km"]
    filled = 0
    for k in key_fields:
        v = row.get(k)
        if v is not None and str(v).strip() not in ["", "nan", "None"]:
            filled += 1

    if filled < 2:
        raise RuntimeError("핵심 필드 부족: 에러 페이지 또는 파싱 실패 가능성")

    return True

In [18]:
# =========================
# 상세 페이지 fetch
# =========================
def fetch_and_save_detail_page(driver, url, html_path, txt_path=None, sleep_sec=2.5, max_retry=2):
    last_error = None

    for attempt in range(1, max_retry + 1):
        try:
            driver.get(url)
            wait_body(driver, sec=12)
            safe_sleep(sleep_sec, sleep_sec + 1.2)
            close_popups(driver)
            safe_sleep(0.5, 1.0)

            current_url = driver.current_url
            html = driver.page_source
            body_text = normalize_text(driver.find_element(By.TAG_NAME, "body").text)

            error_signals = [
                "Out of Memory",
                "페이지를 열기 위한 메모리가 충분하지 않음",
                "이 웹페이지를 표시하는 도중 문제가 발생했습니다",
                "Aw, Snap!",
                "ERR_",
            ]

            if any(sig in html or sig in body_text for sig in error_signals):
                raise RuntimeError("크롬 에러 페이지 감지")

            if "/cars/detail/" not in current_url:
                raise RuntimeError(f"상세페이지 이탈: {current_url}")

            if len(body_text) < 80:
                raise RuntimeError("본문 텍스트가 너무 짧음")

            html_path.parent.mkdir(parents=True, exist_ok=True)
            html_path.write_text(html, encoding="utf-8")

            if txt_path is not None:
                txt_path.write_text(body_text, encoding="utf-8")

            return html

        except Exception as e:
            last_error = e
            print(f"[RETRY {attempt}/{max_retry}] {url} -> {e}")
            safe_sleep(3.0, 5.0)

    raise last_error

In [19]:
# =========================
# 파싱 함수들
# =========================
def parse_meta_description(soup):
    result = {}
    meta = soup.find("meta", attrs={"name": "description"})
    if not meta:
        return result

    content = meta.get("content", "")
    result["연식_메타"] = find_first([r"연식:([^,]+)"], content)
    result["주행거리_메타"] = find_first([r"주행거리:([^,]+)"], content)
    result["연료_메타"] = find_first([r"연료:([^,]+)"], content)
    result["색상_메타"] = find_first([r"색상:([^,]+)"], content)
    result["지역_메타"] = find_first([r"지역:([^,]+?) 중고차"], content)
    return result


def parse_dom_basic(soup):
    result = {
        "차량명_dom": None,
        "모델_dom": None,
        "세부트림_dom": None,
        "연식_원문_dom": None,
        "주행거리_원문_dom": None,
        "연료_dom": None,
        "차량번호_dom": None,
        "등록번호_dom": None,
        "조회수_dom": None,
        "찜수_dom": None,
        "해시태그_dom": None,
        "실촬영문구_dom": None,
    }

    h3 = soup.find("h3")
    if h3:
        spans = [normalize_text(x.get_text(" ", strip=True)) for x in h3.find_all("span")]
        spans = [x for x in spans if x]
        if len(spans) >= 2:
            result["모델_dom"] = spans[0]
            result["세부트림_dom"] = spans[1]
            result["차량명_dom"] = " ".join(spans).strip()
        else:
            result["차량명_dom"] = normalize_text(h3.get_text(" ", strip=True))

    dl = soup.select_one("dl.ar1Ivd7EgX")
    if dl:
        dts = dl.find_all("dt")
        dds = dl.find_all("dd")
        for dt, dd in zip(dts, dds):
            k = normalize_text(dt.get_text(" ", strip=True))
            v = normalize_text(dd.get_text(" ", strip=True))
            if "연식" in k:
                result["연식_원문_dom"] = v
            elif "주행거리" in k:
                result["주행거리_원문_dom"] = v
            elif "연료" in k:
                result["연료_dom"] = v
            elif "차량번호" in k:
                result["차량번호_dom"] = v

    for li in soup.select("ul.rVigc5A_1H li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("등록번호"):
            result["등록번호_dom"] = txt.replace("등록번호", "").strip()
        elif txt.startswith("조회수"):
            result["조회수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("찜"):
            result["찜수_dom"] = parse_int_from_text(txt)

    tags = [normalize_text(li.get_text(" ", strip=True)) for li in soup.select("ul.kYC8KS_YZ1 li")]
    tags = [x for x in tags if x]
    result["해시태그_dom"] = ",".join(tags) if tags else None

    shot = soup.select_one("p.s8FzQTBVpE")
    if shot:
        result["실촬영문구_dom"] = normalize_text(shot.get_text(" ", strip=True))

    return result


def parse_detail_embedded(html):
    result = {}

    str_patterns = {
        "manufacturerName": [r'"manufacturerName":"([^"]+)"'],
        "modelName": [r'"modelName":"([^"]+)"'],
        "gradeName": [r'"gradeName":"([^"]+)"'],
        "gradeDetailName": [r'"gradeDetailName":"([^"]+)"'],
        "vehicleNo": [r'"vehicleNo":"([^"]+)"'],
        "vin": [r'"vin":"([^"]+)"'],
        "requestUrl": [r'"requestUrl":"([^"]+)"'],
        "dealerName": [r'"dealer":\{"userId":"[^"]+","name":"([^"]+)"'],
        "firmName": [r'"firm":\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterName": [r'"diagnosisCenters":\[\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterPhone": [r'"telephoneNumber":"([^"]+)"'],
        "diagnosisCenterAddress": [r'"address":"([^"]+)"'],
        "pageAccessToken": [r'"pageAccessToken":"([^"]+)"'],
        "yearMonth": [r'"yearMonth":"([^"]+)"'],
        "formYear": [r'"formYear":"([^"]+)"'],
        "fuelName": [r'"fuelName":"([^"]+)"'],
        "colorName": [r'"colorName":"([^"]+)"'],
        "transmissionName": [r'"transmissionName":"([^"]+)"'],
        "bodyName": [r'"bodyName":"([^"]+)"'],
        "oneLineText": [r'"oneLineText":"([^"]+)"'],
    }

    int_patterns = {
        "price": [r'"price":(\d+)'],
        "viewCount": [r'"viewCount":(\d+)'],
        "subscribeCount": [r'"subscribeCount":(\d+)'],
        "vehicleId": [r'"vehicleId":(\d+)'],
        "seizingCount": [r'"seizingCount":(\d+)'],
        "pledgeCount": [r'"pledgeCount":(\d+)'],
        "encarDiagnosis": [r'"encarDiagnosis":(-?\d+)'],
        "encarMeetGo": [r'"encarMeetGo":(-?\d+)'],
        "mileage": [r'"mileage":(\d+)'],
        "displacement": [r'"displacement":(\d+)'],
        "seatCount": [r'"seatCount":(\d+)'],
    }

    for key, pats in str_patterns.items():
        result[key] = find_first(pats, html)

    for key, pats in int_patterns.items():
        result[key] = find_first(pats, html, cast=to_int)

    standard_raw = find_first([r'"standard":(\[[^\]]*\])'], html)
    choice_raw = find_first([r'"choice":(\[[^\]]*\])'], html)
    tuning_raw = find_first([r'"tuning":(\[[^\]]*\])'], html)
    etc_raw = find_first([r'"etc":(\[[^\]]*\]|"[^"]*")'], html)

    result["option_standard_codes"] = parse_json_array_str(standard_raw)
    result["option_choice_codes"] = parse_json_array_str(choice_raw)
    result["option_tuning_codes"] = parse_json_array_str(tuning_raw)

    if etc_raw and etc_raw.startswith("["):
        result["option_etc_values"] = parse_json_array_str(etc_raw)
    elif etc_raw:
        result["option_etc_values"] = [etc_raw.strip('"')]
    else:
        result["option_etc_values"] = []

    return result


def parse_dom_major_options(soup):
    result = {}
    for li in soup.select("ul.dz3qFYruNO li"):
        text = normalize_text(li.get_text(" ", strip=True))
        blind = li.select_one("span.blind")
        status = normalize_text(blind.get_text(" ", strip=True)) if blind else None

        if status:
            name = normalize_text(text.replace(status, ""))
            result[f"주요옵션_{name}"] = 1 if status == "있음" else 0 if status == "없음" else None

    return result


def parse_dom_seller_info(soup):
    result = {
        "판매자상호_dom": None,
        "판매자명_dom": None,
        "판매자유형_dom": None,
        "판매중대수_dom": None,
        "판매완료대수_dom": None,
        "판매자지역_dom": None,
        "종사원증번호_dom": None,
    }

    btn = soup.select_one("button.uPEfVsKnZx")
    if btn:
        brand = btn.select_one("span.ESlvHTih77")
        name = btn.select_one("strong.k3tS4rXdrQ")
        seller_type = btn.select_one("span.xf9ufEfgKR")

        if brand:
            result["판매자상호_dom"] = normalize_text(brand.get_text(" ", strip=True))
        if name:
            result["판매자명_dom"] = normalize_text(name.get_text(" ", strip=True))
        if seller_type:
            result["판매자유형_dom"] = normalize_text(seller_type.get_text(" ", strip=True))

    lis = soup.select("ul.VtNR8dNHOS li")
    for li in lis:
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("판매중"):
            result["판매중대수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("판매완료"):
            result["판매완료대수_dom"] = parse_int_from_text(txt)
        elif "종사원증번호" in txt:
            m = re.search(r"종사원증번호\s*([A-Z0-9\-]+)", txt)
            if m:
                result["종사원증번호_dom"] = m.group(1)
        elif any(region in txt for region in ["서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산", "세종", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"]):
            result["판매자지역_dom"] = txt

    return result


def parse_dom_extra_flags(soup):
    result = {
        "보험이력공개여부_dom": None,
        "보험이력건수_dom": None,
        "성능점검유형_dom": None,
        "성능점검설명_dom": None,
    }

    for btn in soup.select("button.fSZoGuAX4M"):
        txt = normalize_text(btn.get_text(" ", strip=True))
        if "보험이력" in txt:
            result["보험이력공개여부_dom"] = txt
            m = re.search(r"보험이력\s*(\d+)건", txt)
            if m:
                result["보험이력건수_dom"] = int(m.group(1))
        elif "성능점검내역" in txt:
            result["성능점검유형_dom"] = txt.replace("성능점검내역", "").strip()

    perf_desc = soup.select_one("div.Hs3feBPsTj p.lGhsYmQaGE")
    if perf_desc:
        result["성능점검설명_dom"] = normalize_text(perf_desc.get_text(" ", strip=True))

    return result


def parse_damage_detail(soup):
    result = {
        "교환_개수": None,
        "판금_개수": None,
        "부식_여부": None,
        "성능기록부_raw": None,
    }

    items = []
    for li in soup.select("ul.wB0X7nC0cq li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt:
            items.append(txt)

        if "교환" in txt:
            if "없음" in txt:
                result["교환_개수"] = 0
            else:
                m = re.search(r"교환\s*(\d+)", txt)
                if m:
                    result["교환_개수"] = int(m.group(1))

        elif "판금" in txt:
            if "없음" in txt:
                result["판금_개수"] = 0
            else:
                m = re.search(r"판금\s*(\d+)", txt)
                if m:
                    result["판금_개수"] = int(m.group(1))

        elif "부식" in txt:
            result["부식_여부"] = 0 if "없음" in txt else 1

    result["성능기록부_raw"] = " || ".join(items) if items else None
    return result


def derive_accident_features(row):
    exchange_cnt = row.get("교환_개수")
    panel_cnt = row.get("판금_개수")
    corrosion = row.get("부식_여부")
    insurance_cnt = row.get("보험이력건수")

    row["교환_여부"] = None if exchange_cnt is None else int(exchange_cnt > 0)
    row["판금_여부"] = None if panel_cnt is None else int(panel_cnt > 0)
    row["외판수리_여부"] = None if (exchange_cnt is None and panel_cnt is None) else int((exchange_cnt or 0) + (panel_cnt or 0) > 0)

    if exchange_cnt is None and panel_cnt is None:
        row["사고강도점수"] = None
    else:
        row["사고강도점수"] = (exchange_cnt or 0) * 2 + (panel_cnt or 0)

    row["보험이력_여부"] = None if insurance_cnt is None else int(insurance_cnt > 0)

    signals = []
    for v in [row["교환_여부"], row["판금_여부"], row["보험이력_여부"]]:
        if v is not None:
            signals.append(v)

    row["사고종합_여부"] = None if not signals else int(any(signals))

    if row["사고강도점수"] is None:
        row["중대사고_추정"] = None
    else:
        row["중대사고_추정"] = int((exchange_cnt or 0) >= 2 or row["사고강도점수"] >= 4)

    row["부식_위험"] = corrosion
    return row


def parse_detail_file(detail_html: str, car_id_hint=None):
    soup = BeautifulSoup(detail_html, "lxml")

    meta = parse_meta_description(soup)
    dom = parse_dom_basic(soup)
    emb = parse_detail_embedded(detail_html)
    major_options = parse_dom_major_options(soup)
    seller_dom = parse_dom_seller_info(soup)
    flags_dom = parse_dom_extra_flags(soup)
    damage_dom = parse_damage_detail(soup)

    model = emb.get("modelName") or dom.get("모델_dom")
    grade = emb.get("gradeName")
    detail = emb.get("gradeDetailName") or dom.get("세부트림_dom")

    if normalize_text(grade) == normalize_text(detail):
        detail = None

    row = {
        "매물ID": str(emb.get("vehicleId") or car_id_hint).strip() if (emb.get("vehicleId") or car_id_hint) else None,
        "제조사": emb.get("manufacturerName"),
        "모델": model,
        "등급명": grade,
        "세부트림": detail,
        "차량명": build_full_trim(model, grade, detail),
        "현재가격_만원": emb.get("price"),
        "조회수": emb.get("viewCount") or dom.get("조회수_dom"),
        "찜수": emb.get("subscribeCount") or dom.get("찜수_dom"),
        "차량번호": emb.get("vehicleNo") or dom.get("차량번호_dom"),
        "VIN": emb.get("vin"),
        "연식_원문": dom.get("연식_원문_dom") or meta.get("연식_메타"),
        "연식": convert_year_from_embedded(
            emb.get("yearMonth"),
            emb.get("formYear"),
            dom.get("연식_원문_dom") or meta.get("연식_메타")
        ),
        "주행거리_원문": dom.get("주행거리_원문_dom") or meta.get("주행거리_메타"),
        "주행거리_km": emb.get("mileage") or parse_int_from_text(dom.get("주행거리_원문_dom") or meta.get("주행거리_메타")),
        "연료": emb.get("fuelName") or dom.get("연료_dom") or meta.get("연료_메타"),
        "색상": emb.get("colorName") or meta.get("색상_메타"),
        "지역": meta.get("지역_메타"),
        "변속기": emb.get("transmissionName"),
        "배기량_cc": emb.get("displacement"),
        "차급": emb.get("bodyName"),
        "좌석수": emb.get("seatCount"),
        "등록번호": dom.get("등록번호_dom"),
        "해시태그": dom.get("해시태그_dom"),
        "실촬영문구": dom.get("실촬영문구_dom"),
        "딜러명": emb.get("dealerName") or seller_dom.get("판매자명_dom"),
        "상사명": emb.get("firmName") or seller_dom.get("판매자상호_dom"),
        "판매자유형": seller_dom.get("판매자유형_dom"),
        "판매중대수": seller_dom.get("판매중대수_dom"),
        "판매완료대수": seller_dom.get("판매완료대수_dom"),
        "판매자지역": seller_dom.get("판매자지역_dom"),
        "종사원증번호": seller_dom.get("종사원증번호_dom"),
        "진단센터명": emb.get("diagnosisCenterName"),
        "진단센터전화": emb.get("diagnosisCenterPhone"),
        "진단센터주소": emb.get("diagnosisCenterAddress"),
        "압류건수": emb.get("seizingCount"),
        "저당건수": emb.get("pledgeCount"),
        "엔카진단여부값": emb.get("encarDiagnosis"),
        "엔카믿고값": emb.get("encarMeetGo"),
        "광고한줄문구": emb.get("oneLineText"),
        "보험이력공개여부": flags_dom.get("보험이력공개여부_dom"),
        "보험이력건수": flags_dom.get("보험이력건수_dom"),
        "성능점검유형": flags_dom.get("성능점검유형_dom"),
        "성능점검설명": flags_dom.get("성능점검설명_dom"),
        "requestUrl": emb.get("requestUrl"),
        "pageAccessToken": emb.get("pageAccessToken"),
        "옵션_기본코드개수": len(emb.get("option_standard_codes", [])),
        "옵션_선택코드개수": len(emb.get("option_choice_codes", [])),
        "옵션_튜닝코드개수": len(emb.get("option_tuning_codes", [])),
        "옵션_기타개수": len(emb.get("option_etc_values", [])),
        "옵션_기본코드": ",".join(emb.get("option_standard_codes", [])) if emb.get("option_standard_codes") else None,
        "옵션_선택코드": ",".join(emb.get("option_choice_codes", [])) if emb.get("option_choice_codes") else None,
        "옵션_튜닝코드": ",".join(emb.get("option_tuning_codes", [])) if emb.get("option_tuning_codes") else None,
        "옵션_기타값": " | ".join(emb.get("option_etc_values", [])) if emb.get("option_etc_values") else None,
    }

    row.update(major_options)
    row.update(damage_dom)
    row = derive_accident_features(row)

    return row

In [ ]:
# =========================
# main
# =========================
def main():
    HTML_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    seen_ids = load_seen_ids()
    print(f"[INFO] 기존 저장 매물ID 수: {len(seen_ids)}")

    if len(seen_ids) >= TARGET_TOTAL:
        print(f"[INFO] 이미 목표 건수({TARGET_TOTAL})에 도달했습니다.")
        return

    driver = setup_driver(headless=HEADLESS)
    run_success_count = 0
    run_error_count = 0
    processed_since_restart = 0

    try:
        detail_links = collect_kia_links(
            driver=driver,
            seen_ids=seen_ids,
            target_total=TARGET_TOTAL,
            max_pages=MAX_LISTING_PAGES,
            max_scrolls_per_page=MAX_SCROLLS_PER_PAGE,
            buffer_links=NEW_LINK_BUFFER
        )

        print(f"\n[INFO] 이번 실행에서 확보한 신규 상세링크 수: {len(detail_links)}")

        for idx, detail_url in enumerate(detail_links, start=1):
            car_id = extract_car_id(detail_url)

            if not car_id:
                err = {
                    "idx": idx,
                    "매물ID": None,
                    "상세링크": detail_url,
                    "error": "매물ID 추출 실패"
                }
                print(f"[ERROR] idx={idx}: 매물ID 추출 실패")
                append_jsonl(err, ERROR_LOG_JSONL)
                run_error_count += 1
                continue

            car_id = str(car_id).strip()

            if car_id in seen_ids:
                print(f"[SKIP] 이미 수집됨: {car_id}")
                continue

            if processed_since_restart >= RESTART_EVERY:
                print(f"[INFO] 드라이버 재시작 ({processed_since_restart}건 처리)")
                driver = restart_driver(driver, headless=HEADLESS)
                processed_since_restart = 0
                safe_sleep(2.0, 3.0)

            print(f"\n[{idx}/{len(detail_links)}] 매물ID={car_id}")

            try:
                car_dir = HTML_DIR / car_id
                car_dir.mkdir(parents=True, exist_ok=True)

                detail_html = fetch_and_save_detail_page(
                    driver=driver,
                    url=detail_url,
                    html_path=car_dir / "detail.html",
                    txt_path=car_dir / "detail.txt"
                )

                row = parse_detail_file(detail_html, car_id_hint=car_id)
                row["상세링크"] = detail_url
                row["수집시각"] = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")

                validate_row(row)

                append_jsonl(row, SUCCESS_LOG_JSONL)
                seen_ids.add(car_id)

                run_success_count += 1
                processed_since_restart += 1

                print(f"[OK] {car_id} 완료 / 이번 실행 성공 {run_success_count}건 / 누적 {len(seen_ids)}건")

                if run_success_count % SAVE_EVERY == 0:
                    print(f"[SAVE] 성공 {run_success_count}건 -> CSV 재생성")
                    rebuild_batch_csv()
                    rebuild_error_csv()
                    export_final_json()

                if len(seen_ids) >= TARGET_TOTAL:
                    print(f"[INFO] 누적 {len(seen_ids)}건 도달 -> 목표 완료")
                    break

                safe_sleep(2.0, 3.5)

            except Exception as e:
                err = {
                    "idx": idx,
                    "매물ID": car_id,
                    "상세링크": detail_url,
                    "error": str(e),
                    "수집시각": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
                }
                print(f"[ERROR] {car_id}: {e}")
                append_jsonl(err, ERROR_LOG_JSONL)

                run_error_count += 1
                processed_since_restart += 1
                safe_sleep(2.5, 4.0)

    finally:
        try:
            driver.quit()
        except Exception:
            pass

    final_df = rebuild_batch_csv()
    error_df = rebuild_error_csv()
    export_final_json()

    print("\n===== 종료 요약 =====")
    print(f"이번 실행 성공 건수: {run_success_count}")
    print(f"이번 실행 에러 건수: {run_error_count}")
    print(f"최종 누적 건수: {0 if final_df.empty else len(final_df)}")
    print(f"최종 에러 건수: {0 if error_df.empty else len(error_df)}")
    print(f"저장 완료: {BATCH_CSV}")
    print(f"저장 완료: {ERROR_CSV}")
    print(f"저장 완료: {FINAL_JSON}")
    print(f"HTML 저장 폴더: {HTML_DIR.resolve()}")

    if not final_df.empty:
        show_cols = [
            "매물ID", "차량명", "현재가격_만원",
            "교환_개수", "판금_개수", "부식_여부",
            "보험이력건수", "사고강도점수", "사고종합_여부"
        ]
        show_cols = [c for c in show_cols if c in final_df.columns]
        print("\n[HEAD]")
        print(final_df[show_cols].head())


if __name__ == "__main__":
    pass

[INFO] 기존 저장 매물ID 수: 767
[INFO] 현재 누적 건수: 767
[INFO] 이번 실행에서 필요한 최소 신규 건수: 4233
[INFO] 이번 실행에서 확보할 링크 목표(버퍼 포함): 4383

[LIST PAGE 1]
  [PAGE 1] 현재 페이지 링크 수: 250
  [PAGE 1] 신규 링크 추가 수: 238
  [TOTAL] 누적 신규 링크 수: 238

[LIST PAGE 2]
  [PAGE 2] 현재 페이지 링크 수: 200
  [PAGE 2] 신규 링크 추가 수: 200
  [TOTAL] 누적 신규 링크 수: 438

[LIST PAGE 3]
  [PAGE 3] 현재 페이지 링크 수: 200
  [PAGE 3] 신규 링크 추가 수: 196
  [TOTAL] 누적 신규 링크 수: 634

[LIST PAGE 4]
  [PAGE 4] 현재 페이지 링크 수: 200
  [PAGE 4] 신규 링크 추가 수: 200
  [TOTAL] 누적 신규 링크 수: 834

[LIST PAGE 5]
  [PAGE 5] 현재 페이지 링크 수: 200
  [PAGE 5] 신규 링크 추가 수: 198
  [TOTAL] 누적 신규 링크 수: 1032
[INFO] 페이지 5 이후로 이동하지 못해 목록 수집 종료

[INFO] 이번 실행에서 확보한 신규 상세링크 수: 1032

[1/1032] 매물ID=41015843
[OK] 41015843 완료 / 이번 실행 성공 1건 / 누적 768건

[2/1032] 매물ID=41140726
[OK] 41140726 완료 / 이번 실행 성공 2건 / 누적 769건

[3/1032] 매물ID=40409120
[OK] 40409120 완료 / 이번 실행 성공 3건 / 누적 770건

[4/1032] 매물ID=41509806
[OK] 41509806 완료 / 이번 실행 성공 4건 / 누적 771건

[5/1032] 매물ID=41586917
[OK] 41586917 완료 / 이번 실행 성공 5건 / 누적 772건

[6/10

In [12]:
print("kernel check")

kernel check


In [13]:
TARGET_N

NameError: name 'TARGET_N' is not defined

In [20]:
# =========================
# page 5부터 이어받기 전용
# 기존 main() 실행 대신 이 셀만 실행
# =========================

TARGET_TOTAL = 5000
START_LIST_PAGE = 5

# 부족분보다 조금 더 링크를 확보해서 에러 발생 시에도 5000건에 최대한 맞추기
EXTRA_LINK_BUFFER = globals().get("NEW_LINK_BUFFER", 300)

# 리스트 페이지 탐색 안전장치
MAX_LIST_PAGE_CONTINUE = 5000
MAX_EMPTY_PAGES_CONTINUE = 50
LIST_WAIT_SEC = 2.0


def _sleep_short(sec=1.5):
    try:
        safe_sleep(sec, sec + 0.8)
    except Exception:
        time.sleep(sec)


def build_list_url_for_page(page: int) -> str:
    if re.search(r'([?&]page=)\d+', SEARCH_URL):
        return re.sub(
            r'([?&]page=)\d+',
            lambda m: f"{m.group(1)}{page}",
            SEARCH_URL,
            count=1
        )

    sep = "&" if "?" in SEARCH_URL else "?"
    return f"{SEARCH_URL}{sep}page={page}"


def extract_detail_targets_from_html(html: str):
    html = html.replace("&amp;", "&")

    patterns = [
        # encar 일반 상세
        re.compile(r'((?:https://www\.encar\.com)?/dc/dc_cardetailview\.do[^"\']*?\bcarid=(\d+)[^"\']*)'),
        # fem 상세
        re.compile(r'((?:https://fem\.encar\.com)?/cars/detail/(\d+)[^"\']*)')
    ]

    results = []
    seen_ids = set()

    for pattern in patterns:
        for full_url, car_id in pattern.findall(html):
            car_id = str(car_id).strip()

            if not car_id or car_id in seen_ids:
                continue

            if full_url.startswith("/dc/"):
                full_url = "https://www.encar.com" + full_url
            elif full_url.startswith("/cars/detail/"):
                full_url = "https://fem.encar.com" + full_url

            seen_ids.add(car_id)
            results.append((car_id, full_url))

    return results


def scroll_list_page(driver, rounds=None):
    rounds = rounds or globals().get("MAX_SCROLLS_PER_PAGE", 8)

    last_height = 0
    no_change_count = 0

    for _ in range(rounds):
        try:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        except Exception:
            pass

        _sleep_short(1.0)

        try:
            new_height = driver.execute_script("return document.body.scrollHeight")
        except Exception:
            new_height = last_height

        if new_height == last_height:
            no_change_count += 1
        else:
            no_change_count = 0

        last_height = new_height

        if no_change_count >= 2:
            break


def collect_kia_links_from_page5(driver, seen_ids, target_total, start_page=5):
    remain_needed = max(0, target_total - len(seen_ids))
    collect_goal = remain_needed + EXTRA_LINK_BUFFER

    if remain_needed <= 0:
        print(f"[INFO] 이미 목표 건수({target_total})에 도달했습니다.")
        return []

    detail_links = []
    new_ids = set()

    page = start_page
    empty_streak = 0

    print(f"[INFO] 페이지 {start_page}부터 신규 상세링크 수집 시작")
    print(f"[INFO] 현재 누적 {len(seen_ids)}건 / 추가 필요 {remain_needed}건 / 링크 확보 목표 {collect_goal}건")

    while len(detail_links) < collect_goal and page <= MAX_LIST_PAGE_CONTINUE:
        list_url = build_list_url_for_page(page)
        print(f"\n[LIST PAGE {page}] {list_url}")

        driver.get(list_url)
        _sleep_short(LIST_WAIT_SEC)

        try:
            WebDriverWait(driver, 8).until(
                lambda d: ("carid=" in d.page_source) or ("/cars/detail/" in d.page_source)
            )
        except Exception:
            pass

        scroll_list_page(driver)

        html = driver.page_source
        targets = extract_detail_targets_from_html(html)

        page_new = 0

        for car_id, detail_url in targets:
            if car_id in seen_ids:
                continue
            if car_id in new_ids:
                continue

            new_ids.add(car_id)
            detail_links.append(detail_url)
            page_new += 1

            if len(detail_links) >= collect_goal:
                break

        print(f"  후보 {len(targets)}개 / 신규 {page_new}개 / 누적 신규링크 {len(detail_links)}/{collect_goal}")

        if page_new == 0:
            empty_streak += 1
            print(f"  [NO NEW] 연속 빈 페이지 {empty_streak}/{MAX_EMPTY_PAGES_CONTINUE}")
        else:
            empty_streak = 0

        if empty_streak >= MAX_EMPTY_PAGES_CONTINUE:
            print("\n[STOP] 새 매물이 연속으로 나오지 않아 리스트 페이지 수집을 종료합니다.")
            break

        page += 1

    return detail_links


def main_continue_from_page5():
    HTML_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    seen_ids = load_seen_ids()
    print(f"[INFO] 기존 저장 매물ID 수: {len(seen_ids)}")

    if len(seen_ids) >= TARGET_TOTAL:
        print(f"[INFO] 이미 목표 건수({TARGET_TOTAL})에 도달했습니다.")
        return

    driver = setup_driver(headless=HEADLESS)
    run_success_count = 0
    run_error_count = 0
    processed_since_restart = 0

    try:
        detail_links = collect_kia_links_from_page5(
            driver=driver,
            seen_ids=seen_ids,
            target_total=TARGET_TOTAL,
            start_page=START_LIST_PAGE
        )

        print(f"\n[INFO] 이번 실행에서 확보한 신규 상세링크 수: {len(detail_links)}")

        if not detail_links:
            print("[INFO] 신규 상세링크가 없어 종료합니다.")
            return

        for idx, detail_url in enumerate(detail_links, start=1):
            car_id = extract_car_id(detail_url)

            if not car_id:
                err = {
                    "idx": idx,
                    "매물ID": None,
                    "상세링크": detail_url,
                    "error": "매물ID 추출 실패"
                }
                print(f"[ERROR] idx={idx}: 매물ID 추출 실패")
                append_jsonl(err, ERROR_LOG_JSONL)
                run_error_count += 1
                continue

            car_id = str(car_id).strip()

            if car_id in seen_ids:
                print(f"[SKIP] 이미 수집됨: {car_id}")
                continue

            if processed_since_restart >= RESTART_EVERY:
                print(f"[INFO] 드라이버 재시작 ({processed_since_restart}건 처리)")
                driver = restart_driver(driver, headless=HEADLESS)
                processed_since_restart = 0
                _sleep_short(2.0)

            print(f"\n[{idx}/{len(detail_links)}] 매물ID={car_id}")

            try:
                car_dir = HTML_DIR / car_id
                car_dir.mkdir(parents=True, exist_ok=True)

                detail_html = fetch_and_save_detail_page(
                    driver=driver,
                    url=detail_url,
                    html_path=car_dir / "detail.html",
                    txt_path=car_dir / "detail.txt"
                )

                row = parse_detail_file(detail_html, car_id_hint=car_id)
                row["상세링크"] = detail_url
                row["수집시각"] = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")

                validate_row(row)

                append_jsonl(row, SUCCESS_LOG_JSONL)
                seen_ids.add(car_id)

                run_success_count += 1
                processed_since_restart += 1

                print(f"[OK] {car_id} 완료 / 이번 실행 성공 {run_success_count}건 / 누적 {len(seen_ids)}건")

                if run_success_count % SAVE_EVERY == 0:
                    print(f"[SAVE] 성공 {run_success_count}건 -> CSV 재생성")
                    rebuild_batch_csv()
                    rebuild_error_csv()
                    export_final_json()

                if len(seen_ids) >= TARGET_TOTAL:
                    print(f"[INFO] 누적 {len(seen_ids)}건 도달 -> 목표 완료")
                    break

                _sleep_short(2.0)

            except Exception as e:
                err = {
                    "idx": idx,
                    "매물ID": car_id,
                    "상세링크": detail_url,
                    "error": str(e),
                    "수집시각": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
                }
                print(f"[ERROR] {car_id}: {e}")
                append_jsonl(err, ERROR_LOG_JSONL)

                run_error_count += 1
                processed_since_restart += 1
                _sleep_short(2.5)

    finally:
        try:
            driver.quit()
        except Exception:
            pass

    final_df = rebuild_batch_csv()
    error_df = rebuild_error_csv()
    export_final_json()

    print("\n===== 종료 요약 =====")
    print(f"이번 실행 성공 건수: {run_success_count}")
    print(f"이번 실행 에러 건수: {run_error_count}")
    print(f"최종 누적 건수: {0 if final_df.empty else len(final_df)}")
    print(f"최종 에러 건수: {0 if error_df.empty else len(error_df)}")
    print(f"저장 완료: {BATCH_CSV}")
    print(f"저장 완료: {ERROR_CSV}")
    print(f"저장 완료: {FINAL_JSON}")
    print(f"HTML 저장 폴더: {HTML_DIR.resolve()}")

    if not final_df.empty:
        show_cols = [
            "매물ID", "차량명", "현재가격_만원",
            "교환_개수", "판금_개수", "부식_여부",
            "보험이력건수", "사고강도점수", "사고종합_여부"
        ]
        show_cols = [c for c in show_cols if c in final_df.columns]
        print("\n[HEAD]")
        print(final_df[show_cols].head())


# 실행
main_continue_from_page5()

[INFO] 기존 저장 매물ID 수: 1432
[INFO] 페이지 5부터 신규 상세링크 수집 시작
[INFO] 현재 누적 1432건 / 추가 필요 3568건 / 링크 확보 목표 3718건

[LIST PAGE 5] https://car.encar.com/list/car?page=5&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22(And.Hidden.N._.MultiView2Hidden.N._.(C.CarType.Y._.Manufacturer.%EA%B8%B0%EC%95%84.))%22%2C%22title%22%3A%22%EA%B8%B0%EC%95%84%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D
  후보 200개 / 신규 199개 / 누적 신규링크 199/3718

[LIST PAGE 6] https://car.encar.com/list/car?page=6&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22(And.Hidden.N._.MultiView2Hidden.N._.(C.CarType.Y._.Manufacturer.%EA%B8%B0%EC%95%84.))%22%2C%22title%22%3A%22%EA%B8%B0%EC%95%84%22%2C%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D
  후보 200개 / 신규 102개 / 누적 신규링크 301/3718

[LIST PAGE 7] https://car.encar.com/list/car?page=7&search=%7B%22type%22%3A%22car%22%2C%22action%22%3A%22(And.Hidden.N._.MultiView2Hidden.N._.(C.CarType.Y._.Manuf

KeyboardInterrupt: 

In [21]:
final_df = rebuild_batch_csv()
error_df = rebuild_error_csv()
export_final_json()

print("복구 완료")
print("최종 누적 건수:", 0 if final_df.empty else len(final_df))
print("최종 에러 건수:", 0 if error_df.empty else len(error_df))
print("BATCH_CSV:", BATCH_CSV)
print("ERROR_CSV:", ERROR_CSV)
print("FINAL_JSON:", FINAL_JSON)

복구 완료
최종 누적 건수: 3827
최종 에러 건수: 1
BATCH_CSV: encar_kia_all\output\kia_detail_batch.csv
ERROR_CSV: encar_kia_all\output\kia_detail_errors.csv
FINAL_JSON: encar_kia_all\output\kia_detail_final.json
